In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-lgd-concat-sensitivity'

## 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3==1.24.59

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import boto3
import numpy as np

# lambda handler
def lambda_handler(event, context):
    
    # did removing a feature help performance
    def helped_or_not(flt_performance):
        if flt_performance > 0:
            return 1
        else:
            return 0

    # create tag if in list
    def create_tag(str_feature, list_cols):
        if str_feature in list_cols:
            return 1
        else:
            return 0
    
    # constants
    str_project = '20231010-gen-xii'
    str_model = '03_pricing_lgd'
    str_prefix = f'{str_model}/02_model/02_model/04_batch_sensitivity_analysis/models'
    
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))
    
    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')
    
    # get index files in s3
    print('Getting files in s3...')
    cls_client = boto3.resource('s3')
    cls_bucket = cls_client.Bucket(str_project)
    list_str_filenames = []
    for file in cls_bucket.objects.filter(Prefix=str_prefix):
        # get key
        str_key = file.key
        # make sure it is a .csv
        if '.csv' in str_key:
            # get filename
            str_filename = str_key.split('/')[-1]
            list_str_filenames.append(str_filename)
    print(f'There are {len(list_str_filenames)} files to import')
    
    # iterate and import
    print('Importing files...')
    list_df = []
    for str_filename in list_str_filenames:
        str_uri = f's3://{str_project}/{str_prefix}/{str_filename}'
        df = pd.read_csv(str_uri)
        list_df.append(df)
    
    # create df
    print('Creating data frame...')
    df = pd.concat(list_df)
    del list_df
       
    # logic for sorting
    print('Sorting...')
    if str_eval_metric in ['AUC', 'PRAUC', 'F1']:
        bool_ascending = False
    else:
        bool_ascending = True # works for RMSE
    df.sort_values(by='flt_eval_metric_valid', ascending=bool_ascending, inplace=True)
    
    # get best score from tuning
    print('Getting best score from tuning...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
    df_tuning = pd.read_csv(str_uri)
    flt_best_metric = df_tuning['flt_eval_metric_valid'].iloc[0]
    
    # assign to df
    df['flt_score_to_beat'] = flt_best_metric
    
    # check the features that helped if removed
    print('Checking to see if removing features helped the validation score...')
    # logic
    if str_eval_metric in ['AUC', 'PRAUC', 'F1']:
        # see if valid score is higher than the score to beat
        df['performance'] = df['flt_eval_metric_valid'] - df['flt_score_to_beat']
    else:
        # see if the valid score is higher than the score to beat
        df['performance'] = df['flt_score_to_beat'] - df['flt_eval_metric_valid'] # works for RMSE

    # mark if helped or not
    df['helped'] = df['performance'].apply(helped_or_not)
    # sort
    df.sort_values(by='performance', ascending=False, inplace=True)
    
    # load list of feats in model
    print('Loading the list of features in the model...')
    str_filename = 'df_cols_in_model.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    list_cols_model = list(pd.read_csv(str_uri)['feature'])
    
    # create tag
    print('Creating tag to ensure feature is in model...')
    df['tag_in_model'] = df['feature'].apply(lambda x: create_tag(x, list_cols_model))
    
    # subset
    print('Subsetting output to only those features in the model...')
    df = df[df['tag_in_model'] == 1].copy()
    
    # save to s3
    print('Saving output...')
    # save to s3 as csv
    str_filename = 'df_sensitivity.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/05_lambda_concat_sensitivity/{str_filename}'
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-lgd-concat-sensitivity

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  45.57kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 303d20c679a3
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> d25abc48fcf6
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> 3146759a8e36
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> 7428a9a82206
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> d14dfd13377d
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 07727fc40cd1
Removing intermediate container 07727fc40cd1
 ---> 2cb157e3c4bd
Successfully built 2cb157e3c4bd
Successfully tagged genxii-lgd-concat-sensitivity:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-lgd-concat-sensitivity' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-concat-sensitivity]
aaa9521b45ef: Preparing
7259da6fe633: Preparing
e4a69b79c430: Preparing
d630e2305053: Preparing
3bd433acfe84: Preparing
09b55d38856d: Preparing
e073f5919ae5: Preparing
b3b414f01759: Preparing
8308f08f35ba: Preparing
c8203e562a8c: Preparing
e4a69b79c430: Waiting
d630e2305053: Waiting
3bd433acfe84: Waiting
09b55d38856d: Waiting
e073f5919ae5: Waiting
b3b414f01759: Waiting
8308f08f35ba: Waiting
c8203e562a8c: Waiting
aaa9521b45ef: Pushed
e4a69b79c430: Pushed
d630e2305053: Pushed
e073f5919ae5: Pushed
b3b414f01759: Pushed
8308f08f35ba: Pushed
3bd433acfe84: Pushed
7259da6fe633: Pushed
09b55d38856d: Pushed
c8203e562a8c: Pushed
latest: digest: sha256:024cef9f06938beeec034c1bca493f27f101a81b1e2d7effbfda8e15204d2898 size: 2420


## 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 16:06:07 GMT',
                                      'x-amzn-requestid': '108c1175-fe25-4fb8-af4f-cfe7b140296e'},
                      'HTTPStatusCode': 204,
                      'RequestId': '108c1175-fe25-4fb8-af4f-cfe7b140296e',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '024cef9f06938beeec034c1bca493f27f101a81b1e2d7effbfda8e15204d2898',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-concat-sensitivity',
 'FunctionName': 'genxii-lgd-concat-sensitivity',
 'LastModified': '2024-08-20T16:06:07.642+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-lgd-concat-sensitivity'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1219',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 16:06:08 GMT',
                                      'x-amzn-requestid': '755e7ebe-e289-4efc-a06d-f6ed4f190bdf'},
                      'HTTPStatusCode': 201,
                      'RequestI

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)